# 03 local fast cloud fill

로컬에서 가볍게 `lst_k_filled`를 다시 만든다. Colab GPU는 CVAE 학습에만 쓴다.

기존 heavy/학습 기반 버전과 다른 점:

- baseline은 `nearest clear fill + Gaussian smooth`로 생성
- 구름 내부 불확실성은 clear pixel까지의 거리 기반 `target_weight`로 반영
- 출력은 CVAE가 그대로 읽을 수 있게 `attempt4/cloud_fill_residual_v1/filled_samples/{split}/*.npz`


In [ ]:
# 필요하면 먼저 실행
%pip install -q numpy scipy pandas tqdm matplotlib

from __future__ import annotations

from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.ndimage import distance_transform_edt, gaussian_filter


In [ ]:
SEED = 20260616
BASELINE_SIGMA = 7.0
UNCERTAINTY_DISTANCE_SCALE_PX = 128.0
UNCERTAINTY_DISTANCE_CAP = 2.0
APPLY_LIMIT = None  # debug면 숫자, 전체면 None
OVERWRITE_FILLED = True  

random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'attempt4':
    PROJECT_DIR = PROJECT_DIR.parent
elif (PROJECT_DIR / 'Workspace' / '26.05_MLproj').exists():
    PROJECT_DIR = PROJECT_DIR / 'Workspace' / '26.05_MLproj'
elif not (PROJECT_DIR / 'lst_dataset').exists() and Path('/mnt/c/users/shinh/eigenspace/Workspace/26.05_MLproj').exists():
    PROJECT_DIR = Path('/mnt/c/users/shinh/eigenspace/Workspace/26.05_MLproj')

DATA_DIR = Path(os.environ.get('LST_DATASET_DIR', PROJECT_DIR / 'lst_dataset'))
ATTEMPT_DIR = Path(os.environ.get('ATTEMPT4_DIR', PROJECT_DIR / 'attempt4'))
SAMPLE_DIR = DATA_DIR / 'samples'
RUN_DIR = ATTEMPT_DIR / 'cloud_fill_residual_v1'
FILLED_SAMPLE_DIR = RUN_DIR / 'filled_samples'
RUN_DIR.mkdir(parents=True, exist_ok=True)
FILLED_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR:', DATA_DIR)
print('RUN_DIR:', RUN_DIR)
print('method: smooth_baseline_distance_weight_v1')


In [ ]:
def split_from_path(path):
    return Path(path).parent.name

def scalar(value):
    try:
        return value.item()
    except Exception:
        return value

def scalar_str(value):
    v = scalar(value)
    if isinstance(v, bytes):
        return v.decode('utf-8')
    return str(v)

def as_2d_mask(arr):
    arr = np.asarray(arr)
    if arr.ndim == 3:
        return arr[0].astype(bool)
    return arr.astype(bool)

def valid_lst_mask(arr):
    arr = np.asarray(arr, dtype=np.float32)
    return np.isfinite(arr) & (arr > 150.0) & (arr < 380.0)

def clean_lst_values(arr):
    out = np.asarray(arr, dtype=np.float32).copy()
    out[~valid_lst_mask(out)] = np.nan
    return out

def target_lst_clear(z):
    if 'lst_clear' in z.files:
        return clean_lst_values(z['lst_clear'].astype(np.float32))
    target = z['target'].astype(np.float32)
    idx = [str(x) for x in z['target_channels']].index('lst_k_raw')
    return clean_lst_values(target[idx])

def read_paths():
    report = DATA_DIR / 'raw_repair_quality_report.csv'
    if report.exists():
        df = pd.read_csv(report)
        if 'qc_status' in df.columns and 'file' in df.columns:
            paths = [DATA_DIR / p for p in df.loc[df['qc_status'].eq('HEALTHY'), 'file'].astype(str)]
            paths = sorted([p for p in paths if p.exists()])
            print('using HEALTHY samples from raw_repair_quality_report.csv')
            return paths
    print('repair report not found; using all split npz files')
    return sorted(SAMPLE_DIR.glob('*/*.npz'))

all_paths = read_paths()
train_paths = [p for p in all_paths if split_from_path(p) == 'train']
val_paths = [p for p in all_paths if split_from_path(p) == 'val']
test_paths = [p for p in all_paths if split_from_path(p) == 'test']
all_paths = train_paths + val_paths + test_paths
print('samples train/val/test:', len(train_paths), len(val_paths), len(test_paths), 'total:', len(all_paths))
if not train_paths:
    raise FileNotFoundError('train samples are empty')

with np.load(train_paths[0], allow_pickle=True) as z:
    CONDITION_CHANNELS = [str(x) for x in z['condition_channels']]
    ALL_TARGET_CHANNELS = [str(x) for x in z['target_channels']]
    META_CHANNELS = [str(x) for x in z['meta_channels']]
    GRID_N = int(z['target'].shape[-1])

required = ['elevation', 'slope', 'aspect_sin', 'aspect_cos', 'hillshade_landsat', 'albedo', 'avg_temp', 'wind_u', 'wind_v', 'rain_1h', 'humidity']
missing = [x for x in required if x not in CONDITION_CHANNELS]
if missing:
    raise RuntimeError(f'missing required condition channels: {missing}')
if 'lst_k_raw' not in ALL_TARGET_CHANNELS:
    raise RuntimeError(f'target_channels must contain lst_k_raw, got {ALL_TARGET_CHANNELS}')

banned = ['air', 'pm10', 'pm25', 'o3', 'pressure', 'press', 'pa', 'ndvi', 'nearest_lst']
bad = [c for c in CONDITION_CHANNELS + ALL_TARGET_CHANNELS if any(t in c.lower() for t in banned)]
if bad:
    raise RuntimeError(f'banned channels found: {bad}')

print('condition channels:', len(CONDITION_CHANNELS), CONDITION_CHANNELS)
print('target raw channels:', ALL_TARGET_CHANNELS)
print('meta channels:', META_CHANNELS)


In [ ]:
def nearest_fill_2d(arr):
    arr = arr.astype(np.float32, copy=True)
    mask = np.isfinite(arr)
    if mask.all():
        return arr
    if not mask.any():
        return np.full_like(arr, 300.0, dtype=np.float32)
    idx = distance_transform_edt(~mask, return_distances=False, return_indices=True)
    return arr[tuple(idx)].astype(np.float32)


def smooth_clear_baseline(lst_clear, sigma=BASELINE_SIGMA):
    lst_clear = clean_lst_values(lst_clear)
    clear = valid_lst_mask(lst_clear)
    nearest = nearest_fill_2d(lst_clear)
    if clear.sum() < 2:
        return nearest.astype(np.float32), np.full_like(nearest, 999.0, dtype=np.float32)
    values = np.where(clear, lst_clear, 0.0).astype(np.float32)
    weights = clear.astype(np.float32)
    num = gaussian_filter(values, sigma=sigma, mode='nearest')
    den = gaussian_filter(weights, sigma=sigma, mode='nearest')
    smooth = np.where(den > 1e-6, num / np.maximum(den, 1e-6), nearest)
    # 너무 먼 구름 내부에서 gaussian denominator가 약할 때 nearest를 섞어 안정화한다.
    dist = distance_transform_edt(~clear).astype(np.float32)
    alpha = np.clip(dist / 64.0, 0.0, 1.0)
    baseline = (1.0 - 0.25 * alpha) * smooth + (0.25 * alpha) * nearest
    return baseline.astype(np.float32), dist.astype(np.float32)


def baseline_uncertainty_and_weight(clear_mask, dist):
    distance_factor = np.clip(
        dist / float(UNCERTAINTY_DISTANCE_SCALE_PX),
        0.0,
        float(UNCERTAINTY_DISTANCE_CAP),
    ).astype(np.float32)
    uncertainty = distance_factor.astype(np.float32)
    uncertainty[clear_mask] = 0.0
    weight_filled = (1.0 / (2.0 + distance_factor)).astype(np.float32)
    weight_filled[clear_mask] = 1.0
    return uncertainty.astype(np.float32), weight_filled.astype(np.float32), distance_factor.astype(np.float32)


def load_base_arrays(path):
    with np.load(path, allow_pickle=True) as z:
        lst_clear = target_lst_clear(z)
    clear_mask = valid_lst_mask(lst_clear)
    baseline, dist = smooth_clear_baseline(lst_clear)
    return lst_clear, clear_mask, baseline, dist


In [ ]:
def output_path(sample_path):
    return FILLED_SAMPLE_DIR / sample_path.parent.name / sample_path.name


rows = []
apply_paths = all_paths[:APPLY_LIMIT] if APPLY_LIMIT is not None else all_paths
for i, path in enumerate(tqdm(apply_paths, desc='write smooth baseline filled samples'), start=1):
    out = output_path(path)
    if out.exists() and not OVERWRITE_FILLED:
        rows.append({'file': path.relative_to(DATA_DIR).as_posix(), 'status': 'SKIP_EXISTS', 'output': out.relative_to(RUN_DIR).as_posix()})
        continue
    out.parent.mkdir(parents=True, exist_ok=True)
    with np.load(path, allow_pickle=True) as z:
        data = {k: z[k] for k in z.files}
    lst_clear, clear_mask, baseline, dist = load_base_arrays(path)
    missing = ~clear_mask
    filled = baseline.copy()
    filled[clear_mask] = lst_clear[clear_mask]
    filled = clean_lst_values(filled)
    bad = ~valid_lst_mask(filled)
    fallback = baseline.copy()
    fallback[clear_mask] = lst_clear[clear_mask]
    filled[bad] = fallback[bad]
    filled = clean_lst_values(filled)
    filled_valid = valid_lst_mask(filled)

    raw_target = np.full_like(filled, np.nan, dtype=np.float32)
    raw_target[clear_mask] = lst_clear[clear_mask]
    data['target'] = np.stack([raw_target, filled], axis=0).astype(np.float32)
    data['target_channels'] = np.array(['lst_k_raw', 'lst_k_filled'])
    data['target_mask'] = np.stack([clear_mask, filled_valid], axis=0).astype(np.uint8)
    data['lst_clear'] = raw_target.astype(np.float32)
    data['clear_mask'] = clear_mask[None].astype(np.uint8)
    data['cloud_fill_mask'] = missing[None].astype(np.uint8)

    uncertainty, weight_filled, distance_factor = baseline_uncertainty_and_weight(clear_mask, dist)
    data['cloud_fill_uncertainty'] = uncertainty[None].astype(np.float32)
    weight = np.ones((2, filled.shape[0], filled.shape[1]), dtype=np.float32)
    weight[0] = clear_mask.astype(np.float32)
    weight[1] = weight_filled.astype(np.float32)
    data['target_weight'] = weight.astype(np.float32)
    data['fill_info'] = np.array(json.dumps({
        'method': 'smooth_baseline_distance_weight_v1',
        'baseline': f'nearest_clear_gaussian_sigma_{BASELINE_SIGMA}',
        'uncertainty': 'clipped_clear_distance_factor',
        'uncertainty_distance_scale_px': float(UNCERTAINTY_DISTANCE_SCALE_PX),
        'uncertainty_distance_cap': float(UNCERTAINTY_DISTANCE_CAP),
        'clear_fraction': float(clear_mask.mean()),
        'filled_pixels': int(missing.sum()),
        'source_sample': str(path.relative_to(PROJECT_DIR)) if path.is_relative_to(PROJECT_DIR) else str(path),
    }, ensure_ascii=False))
    np.savez_compressed(out, **data)
    rows.append({
        'file': path.relative_to(DATA_DIR).as_posix(),
        'status': 'OK',
        'output': out.relative_to(RUN_DIR).as_posix(),
        'clear_fraction': float(clear_mask.mean()),
        'cloud_fraction': float(missing.mean()),
        'distance_factor_mean_cloud': float(np.nanmean(distance_factor[missing])) if missing.any() else 0.0,
        'target_weight_mean_cloud': float(np.nanmean(weight_filled[missing])) if missing.any() else 1.0,
        'filled_valid_fraction': float(filled_valid.mean()),
    })
    if i == 1 or i % 100 == 0:
        print(f"[{rows[-1]['status']} {i:04d}/{len(apply_paths)}] {path.parent.name} | {path.name} cloud={missing.mean():.3f}")

manifest = pd.DataFrame(rows)
manifest.to_csv(RUN_DIR / 'filled_sample_manifest.csv', index=False, encoding='utf-8-sig')
print('status counts:', manifest['status'].value_counts().to_dict())
print('outputs:', FILLED_SAMPLE_DIR)
manifest.head()


In [ ]:
manifest = pd.read_csv(RUN_DIR / 'filled_sample_manifest.csv')
print(manifest['status'].value_counts())
print(manifest.groupby(manifest['file'].str.split('/').str[0]).size())
summary_cols = [c for c in ['clear_fraction', 'cloud_fraction', 'distance_factor_mean_cloud', 'target_weight_mean_cloud', 'filled_valid_fraction'] if c in manifest.columns]
if summary_cols:
    display(manifest[summary_cols].describe())

print('\nColab에 올릴 것은 이 폴더다:')
print(FILLED_SAMPLE_DIR)
print('Drive에서는 MyDrive/attempt4/cloud_fill_residual_v1/filled_samples 로 맞추면 02가 그대로 읽는다.')
